In [32]:
import numpy as np
from tensorflow.keras.datasets import mnist

In [41]:
# Data Loading and Pre-processing
(X_train,y_train),(X_test,y_test) = mnist.load_data()

# Normalise
X_train = X_train.reshape(-1,28*28)/255
X_test = X_test.reshape(-1,28*28)/255

# One-hot Encoding
def one_hot(y,k=10):
    oh = np.zeros((y.size,k))
    oh[np.arange(y.size),y]=1
    return oh
Y_train = one_hot(y_train)
Y_test = one_hot(y_test)

In [34]:
# Activations
def relu(z):
    return np.maximum(0,z)

def softmax(z):
    z-=np.max(z,axis=1,keepdims=True)
    exp_z = np.exp(z)
    return exp_z/(np.sum(exp_z,axis=1,keepdims=True))

def relu_grad(z):
    return (z>0)

In [ ]:
# MLP Model
class MLP:
    # Architecture : input layer(784 pixel values) - hidden layer(any size) - output layer(10 probable outcomes)
    def __init__(self,input_layer=784,hidden_layer=256,output_layer=10):
        # He initialisation : For faster convergence
        self.W1 = np.random.randn(input_layer,hidden_layer)*np.sqrt(2/input_layer)
        self.b1 = np.zeros((1,hidden_layer))
        
        self.W2 = np.random.randn(hidden_layer,output_layer)*np.sqrt(2/hidden_layer)
        self.b2 = np.zeros((1,output_layer))
        
    def forward(self,X):
        self.Z1 = X@self.W1 + self.b1
        self.A1 = relu(self.Z1)
        
        self.Z2 = self.A1@self.W2 + self.b2
        self.Y_hat = softmax(self.Z2)
        return self.Y_hat
    
    def backprop(self,X,Y):
        m = X.shape[0]
        
        dL_dZ2 = self.Y_hat - Y # Loss in softmax : (phi - e_y)
        dL_dW2 = (self.A1.T @ dL_dZ2)/m
        dL_db2 = np.sum(dL_dZ2,axis=0,keepdims=True)/m
        
        dL_dA1 = dL_dZ2@self.W2.T
        dL_dZ1 = dL_dA1 * relu_grad(self.Z1)
        dL_dW1 = (X.T@dL_dZ1)/m
        dL_db1 = np.sum(dL_dZ1,axis=0,keepdims=True)/m
        
        return dL_dW1,dL_db1,dL_dW2,dL_db2

    def update(self,dL_dW1,dL_db1,dL_dW2,dL_db2,alpha):
        self.W1-=alpha*dL_dW1
        self.W2-=alpha*dL_dW2
        self.b1-=alpha*dL_db1
        self.b2-=alpha*dL_db2
        
    def predict(self,X):
        probs = self.forward(X)
        return np.argmax(probs,axis=1)        

In [45]:
# Training the Model
model = MLP()
epochs = 30
alpha = 0.01
batch = 64

n = X_train.shape[0]
for j in range(epochs):
    examples = np.random.permutation(n)
    X_train_shuffled = X_train[examples]
    y_train_shuffled = y_train[examples]
    Y_train_shuffled = Y_train[examples]
    
    for i in range(0,n,batch):
        # Mini Batch Gradien Descent
        X_batch = X_train_shuffled[i:i+batch]
        Y_batch = Y_train_shuffled[i:i+batch]
        
        Y_hat = model.forward(X_batch)
        dL_dW1,dL_db1,dL_dW2,dL_db2 = model.backprop(X_batch,Y_batch)
        model.update(dL_dW1,dL_db1,dL_dW2,dL_db2,alpha)
    # Evaluate
    train_preds = model.predict(X_train[:5000])
    train_acc = np.mean(train_preds == y_train[:5000])
    test_preds = model.predict(X_test)
    test_acc = np.mean(test_preds == y_test)

    print(f"Epoch{j+1}")
    print(f"Train Accuracy:{(train_acc)*100:.2f}%")
    print(f"Test Accuracy:{test_acc*100:.2f}%")
    print("-"*50)

Epoch1
Train Accuracy:88.60%
Test Accuracy:88.77%
--------------------------------------------------
Epoch2
Train Accuracy:90.90%
Test Accuracy:90.61%
--------------------------------------------------
Epoch3
Train Accuracy:91.54%
Test Accuracy:91.58%
--------------------------------------------------
Epoch4
Train Accuracy:92.46%
Test Accuracy:92.00%
--------------------------------------------------
Epoch5
Train Accuracy:93.16%
Test Accuracy:92.45%
--------------------------------------------------
Epoch6
Train Accuracy:93.44%
Test Accuracy:92.86%
--------------------------------------------------
Epoch7
Train Accuracy:94.00%
Test Accuracy:93.10%
--------------------------------------------------
Epoch8
Train Accuracy:94.28%
Test Accuracy:93.47%
--------------------------------------------------
Epoch9
Train Accuracy:94.44%
Test Accuracy:93.59%
--------------------------------------------------
Epoch10
Train Accuracy:94.68%
Test Accuracy:93.80%
----------------------------------------

In [ ]:
# 95%+ Accuracy !